# (Obselete)Final Tool: NeuralProphet to Forecast Citywide Rat Sightings for 14 days

Run all the code blocks. This notebook will download the most recent rat sightings data from NYC Open Data's database on rat sightings, filter for MANHATTAN as the location of the rat sightings, and then make a 14 day forecast. The parameters currently saved were obtained from 2models_manhattan.ipynb and the tuning there.

### Set Parameters

In [1]:
parameters = dict()
parameters['apparent_temperature_max'] = 54
parameters['apparent_temperature_min'] = 18
parameters['snowfall_sum'] = 1
parameters['n_lags'] = 59 
parameters['epochs'] = 493 
parameters['learning_rate'] = 0.003214767890388168 
parameters['batch_size'] = 220
parameters['ar_reg'] = 0.5847571241076923


parameters['n_forecasts'] = 14 

params = parameters.copy()

### Import Packages

In [2]:
import requests 
import random
import torch
import numpy as np
import pandas as pd
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
from neuralprophet import NeuralProphet
import logging

/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [3]:
url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

response = requests.get(url)
response.raise_for_status()

with open("../../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv", "wb") as f:
    f.write(response.content)

In [18]:
rs = pd.read_csv('../../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='MANHATTAN']



rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to the last date in rs
full_range = pd.date_range(start='2020-01-01', end=rs['ds'].max())

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

In [19]:
# Weather data
# The forecasted weather values are the naive forecast i.e. we take the last observed day and assume that these are good predictions for the next 14 days.
# Better weather data might help improve the model.


lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

In [20]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [21]:
# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

In [22]:
# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Train the model

In [23]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(



Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01%

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


### Forecast for 14 Days

In [24]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

forecast_vertical = pd.DataFrame({'ds': last_14_dates, 'yhat': last_14_preds.values})

for _, row in forecast_vertical.iterrows():
    print(f"{row['ds'].date()} -- Prediction: {round(row['yhat'])}.")

2026-03-10 -- Prediction: 15.
2026-03-11 -- Prediction: 15.
2026-03-12 -- Prediction: 16.
2026-03-13 -- Prediction: 14.
2026-03-14 -- Prediction: 10.
2026-03-15 -- Prediction: 14.
2026-03-16 -- Prediction: 16.
2026-03-17 -- Prediction: 17.
2026-03-18 -- Prediction: 18.
2026-03-19 -- Prediction: 16.
2026-03-20 -- Prediction: 15.
2026-03-21 -- Prediction: 10.
2026-03-22 -- Prediction: 14.
2026-03-23 -- Prediction: 19.
